## Cargar un Dataset con Pandas

Nota: La funcion **df.info(memory_usage="deep")** nos ayuda a saber la memoria consumida para cargar el dataframe en memoria.

In [ ]:
import pandas as pd

df = pd.read_csv("twitch_es_mar_26.csv", sep="\t")
df.info(memory_usage='deep')


## Cargar solo cabecera del CSV para saber qué columnas nos interesan

El argumento **nrows=0** nos carga únicamente la cabecera del csv.

In [23]:
import pandas as pd

# Carga solo los metadatos (columnas)
df = pd.read_csv('twitch_es_mar_26.csv', sep="\t", nrows=0)
df.info(memory_usage='deep')
# Ver la lista de columnas
print(df.columns.tolist())

<class 'pandas.DataFrame'>
RangeIndex: 0 entries
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   captured_at    0 non-null      object
 1   language       0 non-null      object
 2   position       0 non-null      object
 3   streamer_name  0 non-null      object
 4   streamer_id    0 non-null      object
 5   game_name      0 non-null      object
 6   game_id        0 non-null      object
 7   viewer_count   0 non-null      object
 8   stream_title   0 non-null      object
dtypes: object(9)
memory usage: 132.0 bytes
['captured_at', 'language', 'position', 'streamer_name', 'streamer_id', 'game_name', 'game_id', 'viewer_count', 'stream_title']


## Cargar Solo ciertas columnas

In [ ]:
import pandas as pd

df = pd.read_csv('twitch_es_mar_25.csv', sep="\t", usecols=['captured_at',"position"])
df.info(memory_usage='deep')


## Guardar el Dataset con las columnas que nos interesan

In [ ]:
import pandas as pd

df = pd.read_csv('twitch_es_mar_25.csv', sep="\t", usecols=['captured_at',"position"])
df.info(memory_usage='deep')
df.to_csv("dataset_limpio.csv", sep="\t", index=False)

## Calcular promedios y sumas Agregadas por fecha de captura

In [ ]:
import pandas as pd

df = pd.read_csv('twitch_es_mar_25.csv', sep="\t")

df_resumen = df.groupby('captured_at').agg({
    'viewer_count': ['sum', 'mean'],
    'streamer_id': 'nunique'
})

df_resumen.columns = ['total_viewers', 'avg_viewers', 'unique_streamers']
df_resumen = df_resumen.reset_index()
df_resumen["avg_viewers"] =  df_resumen["avg_viewers"].round(2)
df_resumen.to_csv('resumen_metricas_captura.csv', sep="\t", index=False)
df_resumen.info(memory_usage='deep')

print("¡Archivo generado con éxito!")
print(df_resumen.head())

# Carga por CHUNKS

## Filtrar solo ciertas posiciones del Ranking en cada captura

Cuando un dataset supera la memoria ram disponible, debemos cargarlo y procesarlo por secciones. Cada una de estas secciones se llama "Chunk". 

En la lectura del CSV, se introduce el argumento *chunksize=chunk_size*. Donde chunk_size es el número de filas que vamos a cargar en cada iteración. 

In [ ]:
import pandas as pd

chunk_size = 100000  # Leer de 100k en 100k filas
dataset_salida = "nombre_dataset_salida.csv"

lista_fragmentos = []

for chunk in pd.read_csv("twitch_es_mar_25.csv", sep="\t", chunksize=chunk_size):
    mask = (chunk["position"] >= 0) & (chunk["position"] <= 100)
    chunk_filtrado = chunk[mask]
    if not chunk_filtrado.empty:
        lista_fragmentos.append(chunk_filtrado)

# Unimos todos los trozos filtrados en un solo DataFrame final
df_final = pd.concat(lista_fragmentos, ignore_index=True)
df_final.to_csv(f"{dataset_salida}", sep="\t", quotechar='"', index=False)
print("ok!")

# CONTROL DE MEMORIA CONSUMIDA en DF_FINAL
df_final.info(memory_usage='deep')

**Pregunta**: ¿Existe algún limite de tamaño para los datasets si se cargan por Chunks? 

**Pregunta**: ¿Qué pasa si la "lista_fragmentos" supera nuestra capacidad de RAM?

## Comparamos los datos de 2025 y 2026

In [3]:
import pandas as pd

def procesar_con_metodo_ligero(archivo_entrada, archivo_salida):
    # 1. CARGA SELECTIVA: Solo 3 columnas y tipos de datos optimizados
    # Esto reduce el consumo de RAM en un 90% desde el inicio.
    df = pd.read_csv(
        archivo_entrada, 
        sep="\t", 
        usecols=['captured_at', 'viewer_count', 'streamer_id'],
        dtype={'viewer_count': 'int32', 'streamer_id': 'int64'}
    )

    # 2. NORMALIZACIÓN: Forzamos que los segundos sean "00" para que el groupby no se rompa
    df['captured_at'] = pd.to_datetime(df['captured_at']).dt.floor('15min')

    # 3. LÓGICA DE AGREGACIÓN
    df_resumen = df.groupby('captured_at').agg({
        'viewer_count': ['sum', 'mean'],
        'streamer_id': 'nunique'
    })

    # 4. FORMATEO
    df_resumen.columns = ['total_viewers', 'avg_viewers', 'unique_streamers']
    df_resumen = df_resumen.reset_index()
    df_resumen["avg_viewers"] = df_resumen["avg_viewers"].round(2)

    # 5. EXPORTACIÓN
    df_resumen.to_csv(archivo_salida, sep="\t", index=False)
    
    return df_resumen

# --- EJECUCIÓN PARA AMBOS AÑOS ---

print("Procesando 2025...")
resumen_25 = procesar_con_metodo_ligero('twitch_es_mar_25.csv', 'resumen_2025.csv')

print("Procesando 2026...")
resumen_26 = procesar_con_metodo_ligero('twitch_es_mar_26.csv', 'resumen_2026.csv')

# --- COMPARATIVA FINAL ---
# Creamos la clave sin el año para poder cruzar los datos
resumen_25['id_temporal'] = resumen_25['captured_at'].dt.strftime('%m-%d %H:%M')
resumen_26['id_temporal'] = resumen_26['captured_at'].dt.strftime('%m-%d %H:%M')

df_comparativa = pd.merge(
    resumen_25[['id_temporal', 'total_viewers', 'unique_streamers']], 
    resumen_26[['id_temporal', 'total_viewers', 'unique_streamers']], 
    on='id_temporal', 
    suffixes=('_2025', '_2026')
)

print("\n¡Comparativa lista!")
print(df_comparativa.head())

# --- CÁLCULO DE MÉTRICAS INTERANUALES ---

# 1. Diferencia absoluta y porcentual en Audiencia (Viewers)
df_comparativa['diff_viewers'] = df_comparativa['total_viewers_2026'] - df_comparativa['total_viewers_2025']
df_comparativa['pct_change_viewers'] = (df_comparativa['diff_viewers'] / df_comparativa['total_viewers_2025']) * 100

# 2. Diferencia absoluta y porcentual en Streamers Únicos
df_comparativa['diff_streamers'] = df_comparativa['unique_streamers_2026'] - df_comparativa['unique_streamers_2025']
df_comparativa['pct_change_streamers'] = (df_comparativa['diff_streamers'] / df_comparativa['unique_streamers_2025']) * 100

# 3. Limpieza: Redondear porcentajes para que sean legibles
df_comparativa['pct_change_viewers'] = df_comparativa['pct_change_viewers'].round(2)
df_comparativa['pct_change_streamers'] = df_comparativa['pct_change_streamers'].round(2)

# 4. Exportar el análisis final
df_comparativa.to_csv('analisis_interanual_marzo_25_26.csv', sep="\t", index=False, float_format='%.2f')

# --- VISUALIZACIÓN DE RESULTADOS ---

print("\n--- RESUMEN DEL ANÁLISIS ---")
# Ver los momentos de mayor crecimiento de audiencia
picos = df_comparativa.sort_values(by='diff_viewers', ascending=False).head(5)
print("Top 5 momentos con mayor crecimiento de audiencia en 2026:")
print(picos[['id_temporal', 'total_viewers_2025', 'total_viewers_2026', 'pct_change_viewers']])

# Ver la media de crecimiento de todo el mes
views_mean_growth = df_comparativa['pct_change_viewers'].mean()
streamers_mean_growth = df_comparativa['pct_change_streamers'].mean()

print(f"\nCrecimiento promedio de audiencia en Marzo: {views_mean_growth:.2f}%")
print(f"\nCrecimiento promedio de Streamers en Marzo: {mean_growth:.2f}%")

Procesando 2025...
Procesando 2026...

¡Comparativa lista!
   id_temporal  total_viewers_2025  unique_streamers_2025  total_viewers_2026  \
0  03-01 00:00              225496                  10700              169195   
1  03-01 00:15              221275                  10678              168086   
2  03-01 00:30              223843                  10933              164478   
3  03-01 00:45              221953                  11086              158528   
4  03-01 01:00              224603                  11171              161261   

   unique_streamers_2026  
0                  10807  
1                  10780  
2                  10832  
3                  10963  
4                  11224  

--- RESUMEN DEL ANÁLISIS ---
Top 5 momentos con mayor crecimiento de audiencia en 2026:
     id_temporal  total_viewers_2025  total_viewers_2026  pct_change_viewers
832  03-09 22:30              315228              665969              111.27
829  03-09 21:45              313196             